# STRATA com escala própria nos níveis: Llama-3.1-8B contra k-quants com imatrix (só qualidade)

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*.

Ideia: o nível 0 continua sendo o Q4_K_M com imatrix de verdade, e cada nível guarda, além de 1 bit por peso, uma escala de 4 bits que diz o tamanho do passo (hoje o passo é fixo: metade da célula). No Llama-3.2-1B, com uma escala por bloco de 256 pesos (+0,016 bit/peso por nível), o nível 1 ficou a 5% do Q5_K_M com imatrix e o nível 2 ficou 22% melhor que o Q6_K com imatrix, sem mexer no Q4. Antes de escrever o kernel, este notebook confere se o 8B repete isso.

**Só qualidade:** os pesos de cada nível vão para um GGUF em F16, que o llama.cpp lê sem kernel novo. Em F16 o llama.cpp não arredonda as ativações para 8 bits, então a KL de todo mundo cai; por isso os k-quants também são medidos em F16 (as matrizes desquantizadas). Compare só dentro desta tabela.

Mede, nesta ordem (cada GGUF F16 tem ~16 GB e é apagado depois de medido): Q5_K_M e Q6_K com imatrix; o STRATA de hoje (passo fixo); escala por bloco de 256 (e, com `MEDIR_SUB = True`, por sub-bloco). A parte mais lenta é montar os pesos dos níveis na CPU (~25 min por variante).

In [ ]:
MEDIR_SUB = False        # True mede também a escala por sub-bloco (+0,14 bit; mais ~30 min)
KLD_CHUNKS = 100         # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125     # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 70:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix)$'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Modelo, BF16, imatrix e k-quants com imatrix. Os pesos originais são apagados depois da conversão (os níveis leem o GGUF BF16).

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q4_K_M', 'Q5_K_M', 'Q6_K']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

In [ ]:
kld = {}
def kl_de(path, nome):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    print(nome, 'KL ×1000', round(1000 * kld[nome][0], 2), 'mesmo 1º token', kld[nome][2], '%')

ESC = f'python strata/tests/kernel/strata_escala_nivel.py --bf16 {G}/BF16.gguf --gguf {G}/Q4_K_M_imat.gguf --imatrix {G}/imatrix.gguf'
for q in ['Q5_K_M', 'Q6_K']:
    !python strata/tests/kernel/strata_escala_nivel.py --bf16 {G}/BF16.gguf --gguf {G}/{q}_imat.gguf --modo copia --out /content/e.gguf 2>&1 | grep -v Warning | tail -1
    kl_de('/content/e.gguf', q + '_imat')
    !rm -f /content/e.gguf
VARIANTES = [('fixo', '--modo fixo'), ('bloco', '--modo escala --grupo bloco')]
if MEDIR_SUB:
    VARIANTES.append(('sub', '--modo escala --grupo sub'))
for nome, extra in VARIANTES:
    !{ESC} {extra} --out /content/e1.gguf /content/e2.gguf 2>&1 | grep -v Warning | grep -v '^ *vals'
    kl_de('/content/e1.gguf', f'STRATA_{nome}_N1')
    kl_de('/content/e2.gguf', f'STRATA_{nome}_N2')
    !rm -f /content/e1.gguf /content/e2.gguf
!rm -f /content/kld_base.bin

In [ ]:
from IPython.display import Markdown, display
L = [f'# STRATA com escala própria nos níveis (Llama-3.1-8B, só qualidade, matrizes em F16)', '',
     f'GPU: {GPU}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. '
     'Todas as matrizes das camadas em F16 (k-quants desquantizados), então as ativações não são arredondadas para 8 bits em nenhum modelo.', '',
     '| modelo | bits extras por nível | KL (×1000) | mesmo 1º token |', '|---|---:|---:|---:|']
extra = {'fixo': '0', 'bloco': '~0,016', 'sub': '~0,14'}
for m, v in kld.items():
    e = next((extra[k] for k in extra if f'_{k}_' in m), '')
    L.append(f'| {m} | {e} | {1000 * v[0]:.2f} ± {1000 * v[1]:.2f} | {v[2]:.2f}% |')
L += ['', 'Relação com o k-quant com imatrix do mesmo nível (abaixo de 1 = mais perto do original):', '', '| comparação | KL |', '|---|---:|']
for nome in ['fixo', 'bloco', 'sub']:
    for n, b in [(1, 'Q5_K_M_imat'), (2, 'Q6_K_imat')]:
        a = f'STRATA_{nome}_N{n}'
        if a in kld and b in kld:
            L.append(f'| {a} / {b} | {kld[a][0] / kld[b][0]:.2f} |')
L += ['', 'Llama-3.2-1B no container, mesmo caminho F16 (seção 26): Q5_K_M 11,79, Q6_K 5,37; passo fixo 15,87 / 4,43; bloco 12,41 / 4,20; sub-bloco 11,68 / 4,05.']
os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_escala.md', 'w').write('\n'.join(L) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld},
          open('/content/res/strata_8b_escala.json', 'w'), indent=1)
display(Markdown('\n'.join(L)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_escala.md')
files.download('/content/res/strata_8b_escala.json')